In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')
df_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_order_items = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_geolocation = pd.read_csv('olist_geolocation_dataset.csv')
df_translation = pd.read_csv('product_category_name_translation.csv')

In [ ]:
datasets = {
    'customers': df_customers,
    'orders': df_orders,
    'products': df_products,
    'sellers': df_sellers,
    'order_items': df_order_items,
    'payments': df_order_payments,
    'reviews': df_order_reviews,
    'geolocation': df_geolocation,
    'translation': df_translation
}

for name, df in datasets.items():
    print(f'{name}:')
    print(df.columns.tolist())
    print()

In [ ]:
df_final = df_orders.merge(
    df_customers,
    on='customer_id',
    how='left'
)

df_final = df_final.merge(
    df_order_items,
    on='order_id',
    how='left'
)

df_final = df_final.merge(
    df_products,
    on='product_id',
    how='left'
)

df_final = df_final.merge(
    df_sellers,
    on='seller_id',
    how='left'
)

df_final = df_final.merge(
    df_translation,
    on='product_category_name',
    how='left'
)

In [ ]:
df_final.shape

1. Pedidos realizados entre vendedores e compradores geograficamente mais próximos apresentam menor tempo de entrega?

In [ ]:
#Ja que antes era possivel existir mais de um zip code prefix, ele agrupar pela media
#entao agora so existe uma localizacao aproximada para cada vendedor e comprador
geo = df_geolocation.groupby(
    'geolocation_zip_code_prefix'
).agg(
    latitude=('geolocation_lat', 'mean'),
    longitude=('geolocation_lng', 'mean')
).reset_index()

In [ ]:
df_final = df_final.merge(
    geo,
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)
df_final = df_final.rename(columns={
    'latitude': 'seller_lat',
    'longitude': 'seller_lng'
})

In [ ]:
df_final = df_final.drop(
    columns='geolocation_zip_code_prefix'
)

In [ ]:
df_final = df_final.merge(
    geo,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

In [ ]:
df_final = df_final.rename(columns={
    'latitude': 'customer_lat',
    'longitude': 'customer_lng'
})

In [ ]:
df_final = df_final.drop(
    columns='geolocation_zip_code_prefix'
)

In [ ]:
print(df_final)

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """
    A formula de Haversine calcula a distancia entre dois pontos tendo a 
    latitude e a longitude como parametro
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371 
    
    return c * r

In [ ]:
#Cria uma coluna chamada difference com a distancia entre o comprador e o vendedor 
#apos aplicar a formula de haversine
df_final['difference_km'] = haversine(
    df_final['customer_lng'],
    df_final['customer_lat'],
    df_final['seller_lng'],
    df_final['seller_lat']
)

In [ ]:
#transformando em datetime
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])


In [ ]:
#transformando em datetime
df_final['order_delivered_customer_date'] = pd.to_datetime(df_final['order_delivered_customer_date'])

In [ ]:
df_final['dias_ate_entrega'] = (df_final['order_delivered_customer_date'] 
- df_final['order_purchase_timestamp']).dt.total_seconds() / (60 * 60 * 24)

In [ ]:
print(df_final)

In [ ]:
df_analise = df_final.dropna(
    subset=['difference_km', 'dias_ate_entrega']
).copy()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1) Remove os 1% extremos de cada eixo para evitar que fique muito esticado e comprima a regiao
d = df_analise[['difference_km', 'dias_ate_entrega']].dropna()
d = d[
    (d['difference_km'] <= d['difference_km'].quantile(0.99)) &
    (d['dias_ate_entrega'] <= d['dias_ate_entrega'].quantile(0.99))
]

counts, xedges, yedges = np.histogram2d(
    d['difference_km'], d['dias_ate_entrega'], bins=[40, 30]
)
xc = (xedges[:-1] + xedges[1:]) / 2
yc = (yedges[:-1] + yedges[1:]) / 2
z = counts.T

# 3) Log na cor para evitar que uma cor domine o grafico
z_log = np.where(z > 0, np.log10(z), np.nan)
max_exp = int(np.ceil(np.nanmax(z_log)))

fig = go.Figure(go.Heatmap(
    x=xc, y=yc, z=z_log,
    customdata=z,
    colorscale='YlOrRd',
    hoverongaps=False,
    hovertemplate=(
        'Distância: %{x:.0f} km<br>'
        'Entrega: %{y:.1f} dias<br>'
        'Pedidos: %{customdata:.0f}<extra></extra>'
    ),
    colorbar=dict(
        title='Pedidos<br>(escala log)',
        tickvals=list(range(max_exp + 1)),
        ticktext=[f'{10**i:,}'.replace(',', '.') for i in range(max_exp + 1)],
        ticks='outside'
    )
))

fig.update_layout(
    template='plotly_white',
    title='Distância entre vendedor e comprador x Tempo de entrega',
    title_x=0.5,
    width=1000, height=650,
    xaxis_title='Distância (km)',
    yaxis_title='Tempo de entrega (dias)'
)

fig.show()

a) A Tecnica utilizada foi um heatmap. O grafico foi feito em Python, com Plotly para a visualizacao interativa, Numpy para o calculo das contagens e pandas para a preparacao dos dados. Foi executado no Jupyter notebook.

b) Acredito que a tecnica de heatmap possa ser classificada como complexa porque relaciona duas variaveis quantitativas continuas ao mesmo tempo, o que vai alem de graficos simples de uma dimensao.
    Para que possa ser feita a plotagem, esse tipo de grafico exige agregacao previa por ter uma quantidade de dados muito grande, por isso nao seria possivel fazer com o Scatter Plot, por exemplo, pois haveria problema de sobreposicao, o que dificultaria a visualização.

c) 
         Elemento      /   Variável      /          Tipo        /        Codificação
| Eixo X (posição) | Distância entre vendedor e comprador (`difference_km`) | Quantitativa contínua | Posição horizontal, discretizada em 40 intervalos |
| ---------------- | ------------------------------------------------------ | --------------------- | ------------------------------------------------- |
| Eixo Y (posição) | Tempo de entrega (`dias_ate_entrega`)                  | Quantitativa contínua | Posição vertical, discretizada em 30 intervalos   |

    Decisoes de Mapeamento: 
        O mapeamento foi feito em escala logaritmica porque a maioria dos pedidos se concentra em poucas celulas. Em escala linear, essas celulas dominariam a paleta e as demais nao ficariam visiveis. A barra de cores mostra os valores reais.
        Zero pedidos aparecem com fundo branco, o que separa "sem dados" de "poucos dados".
        O 1% superior de cada eixo foi removido para evitar que os eixos ficassem esticados e comprimissem os dados da regiao de interesse.

d) A visualizacao investiga se a distancia entre vendedor e comprador influencia o tempo de entrega entre pedidos e e-commerce. A hipotese inicial e de que quanto maior a distancia, maior o prazo de entrega, por conta da logistica e do transporte.

    Padroes Observados:
        Maior concentracao observada no canto inferior esquerdo. A regiao se estende ate cerca de 900-1000km, com prazos de 5 a 20 dias, e onde a maior parte do volume se concentra.
        Passando desse ponto, o grafico muda, o que significa menos pedidos. Os pedidos de longa distancia sao raros.
        Para distancias acima de 1000km as linhas inferiores 1 a 5 dias ficam quase vazias. O que e coerente com a hipotese de que a logistica impoe um prazo minimo. 
        
        Interpretacao: A distancia tem influencia, mas e parcial. Afeta o volume, o prazo minimo e a imprevisibilidade, mas sozinha nao determina o prazo. A grande variação em trajetos curtos sugere que outros fatores pesam bastante, como tempo de postagem pelo vendedor, transportadora, região de destino e eventuais atrasos.

    Conclusão. A maior parte dos pedidos é de curta distância e chega em até cerca de 20 dias. A distância aumenta a incerteza e eleva o prazo mínimo, mas não explica sozinha o tempo de entrega, já que há atrasos longos mesmo em trajetos curtos.

e) A visualização usa os recursos interativos nativos do Plotly



2. Categorias especificas de pedidos apresentam menor tempo de entrega?